In [3]:
# ===============================
# REMOVE ALL WARNINGS (IMPORTANT)
# ===============================
import warnings
warnings.simplefilter("ignore")

import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from lime.lime_tabular import LimeTabularExplainer


# ===============================
# LOAD DATA
# ===============================
def load_dataset():
    try:
        # Works in .py script
        base_path = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        # Works in notebook / interactive mode
        base_path = os.getcwd()

    file_path = os.path.join(base_path, "/Users/majjaripranay/Documents/ML_CODES/Clarity_Text_student_teacher_with_glove.xlsx")

    if not os.path.exists(file_path):
        print("ERROR: dataset.xlsx not found at:", file_path)
        return None

    return pd.read_excel(file_path)


# ===============================
# PREPROCESS
# ===============================
def preprocess(data):
    features = [col for col in data.columns if col.startswith("glove_")]

    X = data[features]
    y = data["Label"]

    # Convert to numeric safely
    X = X.apply(pd.to_numeric, errors="coerce")

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    return X, y, encoder


# ===============================
# BUILD MODEL
# ===============================
def build_pipeline():

    base_models = [
        ("rf", RandomForestClassifier(n_estimators=20, random_state=42)),
        ("et", ExtraTreesClassifier(n_estimators=20, random_state=42)),
        ("dt", DecisionTreeClassifier(max_depth=10, random_state=42))
    ]

    final_model = RandomForestClassifier(n_estimators=20, random_state=42)

    stack = StackingClassifier(
        estimators=base_models,
        final_estimator=final_model,
        cv=3,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("variance", VarianceThreshold()),
        ("scaler", StandardScaler()),
        ("model", stack)
    ])

    return pipeline


# ===============================
# TRAIN + EVALUATE
# ===============================
def train_and_evaluate(pipeline, X_train, X_test, y_train, y_test):

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)

    print("\n===== RESULTS =====")
    print("Accuracy:", round(acc, 4))

    print("\nClassification Report:\n")
    print(classification_report(y_test, preds))

    print("Confusion Matrix:\n")
    print(confusion_matrix(y_test, preds))


# ===============================
# LIME
# ===============================
def run_lime(pipeline, X_train, X_test, encoder):

    explainer = LimeTabularExplainer(
        X_train.values,
        feature_names=X_train.columns.tolist(),
        class_names=[str(cls) for cls in encoder.classes_],
        mode="classification"
    )

    exp = explainer.explain_instance(
        X_test.iloc[0].values,
        pipeline.predict_proba
    )

    exp.save_to_file("lime_explanation.html")
    print("LIME saved → lime_explanation.html")


# ===============================
# MAIN
# ===============================
def main():

    data = load_dataset()
    if data is None:
        return

    X, y, encoder = preprocess(data)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipeline = build_pipeline()

    train_and_evaluate(pipeline, X_train, X_test, y_train, y_test)

    run_lime(pipeline, X_train, X_test, encoder)


# ===============================
if __name__ == "__main__":
    main()


===== RESULTS =====
Accuracy: 0.4268

Classification Report:

              precision    recall  f1-score   support

           0       0.14      0.05      0.08        92
           1       0.31      0.35      0.33       213
           2       0.53      0.57      0.55       358

    accuracy                           0.43       663
   macro avg       0.33      0.32      0.32       663
weighted avg       0.40      0.43      0.41       663

Confusion Matrix:

[[  5  27  60]
 [ 15  75 123]
 [ 15 140 203]]
LIME saved → lime_explanation.html
